In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import ipywidgets as widgets
from IPython.display import display, clear_output
import datetime
import random
import uuid

fake = Faker()

class UniversalGenerator:
    """A modular synthetic data generator enforcing relational data integrity."""
    
    def __init__(self):
        self.domains = {
            "Demographics": ["Customer ID", "Name", "Age", "Gender", "Education Level", "Occupation", "Email", "Registered Date"],
            "Health Metrics": ["Height (cm)", "Weight (kg)", "BMI", "Blood Type", "Heart Rate (bpm)"],
            "Geography": ["Country", "City", "State", "Zip Code", "Latitude", "Longitude", "Timezone"],
            "Products": ["SKU", "Product Name", "Category", "Model", "Size", "Color", "Ratings", "Review Count"],
            "Commerce": ["Price", "Quantity", "Discount (%)", "Total", "Order Status", "Order Date", "Shipping Date", "Shipping Carrier", "Payment Method"],
            "IT/System Data": ["IP Address", "MAC Address", "User Agent", "OS", "UUID"]
        }
        
    def _generate_row(self, selected_fields):
        """Generates a single row of logically consistent data."""
        row = {}
        
        status = random.choice(["Delivered", "Processing", "Shipped", "Cancelled"])
        
        if status == "Delivered":
            order_date = fake.date_between(start_date="-1y", end_date="-10d")
            ship_date = order_date + datetime.timedelta(days=random.randint(1, 5))
        elif status == "Shipped":
            order_date = fake.date_between(start_date="-15d", end_date="-3d")
            ship_date = order_date + datetime.timedelta(days=random.randint(1, 2))
        elif status == "Processing":
            order_date = fake.date_between(start_date="-2d", end_date="today")
            ship_date = "Pending"
        else:
            order_date = fake.date_between(start_date="-30d", end_date="today")
            ship_date = "N/A"
            
        registered_date = order_date - datetime.timedelta(days=random.randint(1, 730))

        name = fake.name()
        age = random.randint(18, 85)
        
        if "Customer ID" in selected_fields: row["Customer ID"] = f"CUST-{random.randint(10000, 99999)}"
        if "Name" in selected_fields: row["Name"] = name
        if "Age" in selected_fields: row["Age"] = age
        if "Gender" in selected_fields: row["Gender"] = random.choice(["Male", "Female", "Non-Binary"])
        if "Education Level" in selected_fields: 
            row["Education Level"] = random.choice(["High School", "Bachelor's", "Master's", "PhD"])
        if "Occupation" in selected_fields: row["Occupation"] = fake.job()
        if "Email" in selected_fields: 
            domain = fake.free_email_domain()
            email_slug = name.lower().replace(" ", ".")
            row["Email"] = f"{email_slug}@{domain}"
        if "Registered Date" in selected_fields: row["Registered Date"] = registered_date

        height = random.randint(150, 200)
        weight = random.randint(50, 120)
        bmi = round(weight / ((height / 100) ** 2), 1)
        
        if "Height (cm)" in selected_fields: row["Height (cm)"] = height
        if "Weight (kg)" in selected_fields: row["Weight (kg)"] = weight
        if "BMI" in selected_fields: row["BMI"] = bmi
        if "Blood Type" in selected_fields: 
            row["Blood Type"] = random.choice(["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"])
        if "Heart Rate (bpm)" in selected_fields:
            base_hr = random.randint(60, 85)
            if age > 60 or bmi > 30:
                base_hr = int(base_hr * random.uniform(1.1, 1.25))
            row["Heart Rate (bpm)"] = base_hr

        country = fake.country()
        
        if "Country" in selected_fields: row["Country"] = country
        if "City" in selected_fields: row["City"] = fake.city()
        if "State" in selected_fields: row["State"] = fake.state()
        if "Zip Code" in selected_fields: row["Zip Code"] = fake.postcode()
        if "Latitude" in selected_fields: row["Latitude"] = float(fake.latitude())
        if "Longitude" in selected_fields: row["Longitude"] = float(fake.longitude())
        if "Timezone" in selected_fields: row["Timezone"] = fake.timezone()

        category = random.choice(["Electronics", "Apparel", "Home & Garden"])
        product_adj = fake.word().capitalize()
        
        if category == "Electronics":
            prod_name = f"{product_adj} Smart Device"
            model = f"TechPro {fake.word().capitalize()}"
            size = "N/A"
        elif category == "Apparel":
            prod_name = f"{product_adj} Cotton Garment"
            model = f"Thread {fake.word().capitalize()}"
            size = random.choice(["XS", "S", "M", "L", "XL", "XXL"])
        else:
            prod_name = f"{product_adj} Decor Item"
            model = f"Aura {fake.word().capitalize()}"
            size = random.choice(["Small", "Medium", "Large"])
            
        if "SKU" in selected_fields: 
            row["SKU"] = f"{category[:3].upper()}-{fake.lexify(text='????').upper()}-{random.randint(100, 999)}"
        if "Product Name" in selected_fields: row["Product Name"] = prod_name
        if "Category" in selected_fields: row["Category"] = category
        if "Model" in selected_fields: row["Model"] = model
        if "Size" in selected_fields: row["Size"] = size
        if "Color" in selected_fields: row["Color"] = fake.color_name()
        if "Ratings" in selected_fields: row["Ratings"] = round(random.uniform(1.0, 5.0), 1)
        if "Review Count" in selected_fields: row["Review Count"] = random.randint(0, 500)

        price = round(random.uniform(10.0, 1500.0), 2)
        qty = random.randint(1, 5)
        discount = random.choice([0, 0, 0, 5, 10, 15, 20])
        
        if "Price" in selected_fields: row["Price"] = price
        if "Quantity" in selected_fields: row["Quantity"] = qty
        if "Discount (%)" in selected_fields: row["Discount (%)"] = discount
        if "Total" in selected_fields: row["Total"] = round((price * qty) * (1 - (discount / 100)), 2)
            
        if "Order Status" in selected_fields: row["Order Status"] = status
        if "Order Date" in selected_fields: row["Order Date"] = order_date
        if "Shipping Date" in selected_fields: row["Shipping Date"] = ship_date
        if "Shipping Carrier" in selected_fields: 
            row["Shipping Carrier"] = random.choice(["FedEx", "UPS", "USPS", "DHL"]) if ship_date != "N/A" else "N/A"
                
        if "Payment Method" in selected_fields:
            if country in ["United States", "Canada", "United Kingdom"]:
                row["Payment Method"] = random.choice(["Credit Card", "Apple Pay", "PayPal"])
            elif country in ["Germany", "Netherlands", "France"]:
                row["Payment Method"] = random.choice(["Bank Transfer", "SEPA", "Credit Card"])
            else:
                row["Payment Method"] = random.choice(["Credit Card", "Cash on Delivery", "PayPal"])

        if "IP Address" in selected_fields: row["IP Address"] = fake.ipv4()
        if "MAC Address" in selected_fields: row["MAC Address"] = fake.mac_address()
        if "User Agent" in selected_fields: row["User Agent"] = fake.user_agent()
        if "OS" in selected_fields: 
            row["OS"] = random.choice(["Windows 11", "Windows 10", "macOS", "Linux", "iOS", "Android"])
        if "UUID" in selected_fields: row["UUID"] = str(uuid.uuid4())

        return row

    def generate_dataframe(self, selected_fields, num_rows):
        """Builds the Pandas DataFrame based on selected fields and row count."""
        data = [self._generate_row(selected_fields) for _ in range(num_rows)]
        return pd.DataFrame(data)


generator = UniversalGenerator()
output = widgets.Output()

accordions = []
checkbox_dict = {}

for domain, fields in generator.domains.items():
    vbox_items = []
    for field in fields:
        cb = widgets.Checkbox(value=False, description=field, indent=False)
        checkbox_dict[field] = cb
        vbox_items.append(cb)
    accordions.append(widgets.VBox(vbox_items))

accordion = widgets.Accordion(children=accordions)
for i, title in enumerate(generator.domains.keys()):
    accordion.set_title(i, title)

row_slider = widgets.IntSlider(
    value=100, min=10, max=10000, step=10, 
    description='Row Count:', continuous_update=False
)

generate_btn = widgets.Button(
    description='Generate & Export', 
    button_style='success', 
    icon='check'
)

def on_generate_click(b):
    with output:
        clear_output(wait=True)
        print("Generating... Please wait.")
        
        selected_fields = [field for field, cb in checkbox_dict.items() if cb.value]
        
        if not selected_fields:
            print("Select at least one field from the accordion above.")
            return
            
        df = generator.generate_dataframe(selected_fields, row_slider.value)
        
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
        filename = f"dataset_{timestamp}.csv"
        df.to_csv(filename, index=False)
        
        clear_output(wait=True)
        print(f"Generated {row_slider.value} rows.")
        print(f"File saved to local directory as: {filename}\n")
        display(df.head(10))

generate_btn.on_click(on_generate_click)

ui = widgets.VBox([
    widgets.HTML("<h2>GEn</h2><p>Select your required fields below:</p>"),
    accordion,
    widgets.HTML("<hr>"),
    widgets.HBox([row_slider, generate_btn]),
    output
])

display(ui)